# IrResnet4: обучение с визуализацией и сохранением на Google Drive

Протокол как в оригинале (v1.5.1.72): `structure_smarts`, hidden=72, lr=1e-5, WeightedRandomSampler, split 70/10/20 на `dataset_v003`.

In [ ]:
import subprocess
from pathlib import Path

REPO_DIR = Path('IR_expert_system_3')
if not REPO_DIR.is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/Lamblador/IR_expert_system_3.git', str(REPO_DIR)], check=True)
%cd IR_expert_system_3
!pip install -q -e ".[torch]" iterative-stratification

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
IR_DATA = Path('/content/drive/MyDrive/ir_data')
RUNS_DRIVE = Path('/content/drive/MyDrive/ir_expert_system_3/runs')
RUNS_DRIVE.mkdir(parents=True, exist_ok=True)
print('IR_DATA', IR_DATA.exists(), IR_DATA)
print('RUNS_DRIVE', RUNS_DRIVE)

In [ ]:
from pathlib import Path
from ir_pipeline.config_loader import load_yaml, merge_train_defaults, resolve_dataset_dir, resolve_paths

DATASET_PROFILE = 'auto'  # auto | mini | full
paths_cfg = load_yaml(Path('configs/paths.huggingface.yaml'))
paths_cfg['dataset_profile'] = DATASET_PROFILE
paths = resolve_paths(paths_cfg)
DATASET_DIR = resolve_dataset_dir(paths_cfg, DATASET_PROFILE)
print('dataset:', DATASET_DIR, 'exists:', DATASET_DIR.joinpath('spectra.npz').is_file())

In [ ]:
%matplotlib inline
from pathlib import Path
from ir_pipeline.config_loader import load_yaml, merge_train_defaults
from ir_pipeline.irresnet_train import IrResnetTrainer
from ir_pipeline.train_monitor import IrResnetTrainingPlotter

cfg_name = 'train_irresnet_original.yaml' if DATASET_PROFILE in ('full', 'auto') and 'v003' in str(DATASET_DIR) else 'train_irresnet_mini.yaml'
train_cfg = merge_train_defaults(load_yaml(Path('configs') / cfg_name))
RUN_DIR = Path('runs/colab06_irresnet')
trainer = IrResnetTrainer(
    dataset_dir=DATASET_DIR,
    run_dir=RUN_DIR,
    bands_yaml=paths['bands_config'],
    train_cfg=train_cfg,
)
summary = trainer.fit()
print(summary)

In [ ]:
import json
import shutil
from pathlib import Path
import matplotlib.pyplot as plt

plotter = IrResnetTrainingPlotter.from_train_cfg(RUN_DIR, train_cfg, title='IrResnet4')
hist_path = RUN_DIR / 'irresnet_history.json'
if hist_path.is_file():
    plotter.series = json.loads(hist_path.read_text(encoding='utf-8'))
    plotter.live_plot = True
    plotter._clear_and_plot(len(plotter.series.get('train_loss', [])), int(train_cfg.get('torch_epochs', 30)))
else:
    img = RUN_DIR / 'irresnet_training_curve.png'
    if img.is_file():
        plt.imshow(plt.imread(img))
        plt.axis('off')
        plt.show()

In [ ]:
import shutil
from pathlib import Path

run_name = RUN_DIR.name + '_' + str(DATASET_DIR.name)
dest = RUNS_DRIVE / run_name
if dest.exists():
    shutil.rmtree(dest)
shutil.copytree(RUN_DIR, dest)
zip_path = RUNS_DRIVE / f'{run_name}.zip'
shutil.make_archive(str(RUNS_DRIVE / run_name), 'zip', RUN_DIR)
print('Saved to Drive:', dest)
print('Zip:', zip_path.with_suffix('.zip'))